In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

In [ ]:
# Hyperparameters
input_size = 28 * 28  #28*28 pixels = every image flattened into 1D vector of 784 values
num_classes = 10
num_epochs = 5
batch_size = 100  #feed 100 images at once into the model
learning_rate = 0.001

In [ ]:
# MNIST dataset (images and labels)
train_dataset = torchvision.datasets.MNIST(root='../../data',
                                           train=True,
                                           transform=transforms.ToTensor(),
                                           download=True)

test_dataset = torchvision.datasets.MNIST(root='../../data',
                                          train=False,
                                          transform=transforms.ToTensor())

##MNIST images are PIL images, converted into tensors


In [ ]:
# Data loader (input pipeline): divides dataset into batches
# shuffles the training dataset every epoch
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=False)


In [ ]:
# Logistic regression model
model = nn.Linear(input_size, num_classes)

In [ ]:
# Loss and optimizer

# CrossEntropyLoss computes softmax internally (logits->softmax->cross entropy loss)
criterion = nn.CrossEntropyLoss()

# SGD updates weights and biases using gradients computed during backpropagation
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
# Train the model
total_step = len(train_loader)  # 600 batches so total_step = 600

for epoch in range(num_epochs):
    # images.shape = [100,1,28,28] 100 images, 1 channel, 28*28 pixels
    for i, (images, labels) in enumerate(train_loader):
        # Reshape images to (batch_size, input_size) = [100, 784]
        images = images.reshape(-1, input_size)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad() #clears gradients from previous iteration
        loss.backward() #computes gradients of every parameter
        optimizer.step() #updates weights using SGD

        if (i+1) % 100 == 0:
            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'
                   .format(epoch+1, num_epochs, i+1, total_step, loss.item()))


In [ ]:

# Test the model
# In test phase, we don't need to compute gradients (for memory efficiency)
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.reshape(-1, input_size) #flatten the image
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1) #select index of class with the highest score
        total += labels.size(0)  #count total images processed
        correct += (predicted == labels).sum()  #count correct predictions

    print('Accuracy of the model on the 10000 test images: {} %'.format(100 * correct / total))

In [ ]:
# Save the trained model parameters
torch.save(model.state_dict(), 'model.ckpt')